In [1]:
# ============================================================
# MATMINER: CHEMICAL FORMULA → NUMERICAL FEATURE MATRIX
# ============================================================

import pandas as pd
import numpy as np

from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

# ============================================================
# STEP 1: LOAD CLEANED DATASET
# ============================================================

INPUT_FILE = "materials_project_cleaning/materials_cleaned.csv"

df = pd.read_csv(INPUT_FILE)

print("=" * 70)
print("MATMINER CHEMICAL FORMULA FEATURE GENERATION")
print("=" * 70)

print("\nDataset loaded:")
print("Rows    :", len(df))
print("Columns :", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# STEP 2: FIND FORMULA COLUMN
# ============================================================

possible_formula_columns = [
    "formula",
    "formula_pretty",
    "pretty_formula",
    "composition",
    "chemical_formula",
    "formula_reduced",
    "reduced_formula"
]

formula_col = None

for col in possible_formula_columns:

    if col in df.columns:
        formula_col = col
        break


if formula_col is None:

    raise ValueError(
        "Chemical formula column was not found.\n"
        "Available columns are:\n"
        + "\n".join(df.columns)
    )


print("\nFormula column detected:")
print(formula_col)


# ============================================================
# STEP 3: CONVERT FORMULA TO PYMATGEN COMPOSITION
# ============================================================

print("\nConverting formulas to pymatgen Composition objects...")

def convert_to_composition(formula):

    try:

        if pd.isna(formula):
            return None

        return Composition(str(formula))

    except Exception:

        return None


df["composition"] = (
    df[formula_col]
    .apply(convert_to_composition)
)


# ============================================================
# STEP 4: CHECK INVALID COMPOSITIONS
# ============================================================

invalid_compositions = (
    df["composition"].isna().sum()
)

print(
    "Invalid compositions:",
    invalid_compositions
)

if invalid_compositions > 0:

    print(
        "\nRemoving invalid compositions..."
    )

    df = df.dropna(
        subset=["composition"]
    ).reset_index(drop=True)


# ============================================================
# STEP 5: CREATE MATMINER FEATURIZER
# ============================================================

print("\nCreating Matminer ElementProperty featurizer...")

featurizer = ElementProperty.from_preset(
    "magpie"
)

print("Matminer featurizer created successfully.")


# ============================================================
# STEP 6: GENERATE NUMERICAL FEATURES
# ============================================================

print("\nGenerating numerical features...")
print("This may take some time for a large dataset...")

df_features = featurizer.featurize_dataframe(
    df,
    col_id="composition",
    ignore_errors=True
)


# ============================================================
# STEP 7: DISPLAY GENERATED FEATURES
# ============================================================

print("\nFeature generation completed!")

print(
    "Dataset shape after feature generation:",
    df_features.shape
)

print("\nFirst 10 columns:")

print(
    df_features.columns[:10].tolist()
)


# ============================================================
# STEP 8: IDENTIFY MATMINER FEATURES
# ============================================================

feature_columns = (
    featurizer.feature_labels()
)

print(
    "\nNumber of numerical features generated:",
    len(feature_columns)
)


# ============================================================
# STEP 9: CREATE NUMERICAL MATRIX
# ============================================================

X = df_features[
    feature_columns
].copy()

print("\nNumerical matrix X:")
print("Rows    :", X.shape[0])
print("Features:", X.shape[1])


# ============================================================
# STEP 10: CHECK NaN VALUES
# ============================================================

nan_count = X.isna().sum().sum()

print(
    "\nTotal NaN values in feature matrix:",
    nan_count
)


# ============================================================
# STEP 11: REMOVE FEATURES THAT ARE COMPLETELY NaN
# ============================================================

all_nan_columns = X.columns[
    X.isna().all()
].tolist()

if len(all_nan_columns) > 0:

    print(
        "\nRemoving completely empty features:",
        len(all_nan_columns)
    )

    X = X.drop(
        columns=all_nan_columns
    )


# ============================================================
# STEP 12: HANDLE REMAINING NaN VALUES
# ============================================================

# Replace infinite values
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

# Fill remaining missing numerical values
X = X.fillna(
    X.median(numeric_only=True)
)

# Any columns that still contain NaN
# are removed as a final safeguard.
remaining_nan_columns = X.columns[
    X.isna().any()
].tolist()

if len(remaining_nan_columns) > 0:

    print(
        "Removing columns with remaining NaN:",
        len(remaining_nan_columns)
    )

    X = X.drop(
        columns=remaining_nan_columns
    )


# ============================================================
# STEP 13: FINAL NUMERICAL MATRIX CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL NUMERICAL MATRIX")
print("=" * 70)

print(
    "Rows:",
    X.shape[0]
)

print(
    "Features:",
    X.shape[1]
)

print(
    "NaN values:",
    X.isna().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(X.select_dtypes(
        include=np.number
    )).sum().sum()
)


# ============================================================
# STEP 14: SAVE NUMERICAL MATRIX
# ============================================================

OUTPUT_FILE = (
    "materials_project_cleaning/"
    "matminer_magpie_features.csv"
)

X.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nNumerical feature matrix saved:")
print(OUTPUT_FILE)


# ============================================================
# STEP 15: SAVE DATASET WITH ORIGINAL INFORMATION
# ============================================================

FULL_OUTPUT_FILE = (
    "materials_project_cleaning/"
    "materials_with_magpie_features.csv"
)

# Remove Composition object because it is not
# convenient for CSV storage.

df_features[
    [formula_col]
    + [
        col for col in feature_columns
        if col in df_features.columns
    ]
].to_csv(
    FULL_OUTPUT_FILE,
    index=False
)

print(
    "Full formula + feature dataset saved:"
)

print(FULL_OUTPUT_FILE)


# ============================================================
# STEP 16: PRINT SAMPLE
# ============================================================

print("\nSample numerical matrix:")

display(
    X.head()
)

MATMINER CHEMICAL FORMULA FEATURE GENERATION

Dataset loaded:
Rows    : 10000
Columns : 5

Columns:
['material_id', 'chemical_formula', 'crystal_system', 'space_group', 'full_structure']

Formula column detected:
chemical_formula

Converting formulas to pymatgen Composition objects...
Invalid compositions: 0

Creating Matminer ElementProperty featurizer...
Matminer featurizer created successfully.

Generating numerical features...
This may take some time for a large dataset...


ElementProperty:   0%|          | 0/10000 [00:00<?, ?it/s]


Feature generation completed!
Dataset shape after feature generation: (10000, 138)

First 10 columns:
['material_id', 'chemical_formula', 'crystal_system', 'space_group', 'full_structure', 'composition', 'MagpieData minimum Number', 'MagpieData maximum Number', 'MagpieData range Number', 'MagpieData mean Number']

Number of numerical features generated: 132

Numerical matrix X:
Rows    : 10000
Features: 132

Total NaN values in feature matrix: 0

FINAL NUMERICAL MATRIX
Rows: 10000
Features: 132
NaN values: 0
Infinite values: 0

Numerical feature matrix saved:
materials_project_cleaning/matminer_magpie_features.csv
Full formula + feature dataset saved:
materials_project_cleaning/materials_with_magpie_features.csv

Sample numerical matrix:


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,6.0,20.0,14.0,9.200000,2.720000,8.0,7.0,87.0,80.0,75.100000,...,0.0,0.0,0.0,0.0,12.0,225.0,213.0,87.900000,91.08000,12.0
1,6.0,20.0,14.0,9.466667,3.146667,8.0,7.0,87.0,80.0,73.066667,...,0.0,0.0,0.0,0.0,12.0,225.0,213.0,88.933333,92.32000,12.0
2,6.0,20.0,14.0,9.200000,2.720000,8.0,7.0,87.0,80.0,75.100000,...,0.0,0.0,0.0,0.0,12.0,225.0,213.0,87.900000,91.08000,12.0
3,3.0,17.0,14.0,12.823529,4.913495,17.0,1.0,94.0,93.0,67.529412,...,0.0,0.0,0.0,0.0,64.0,229.0,165.0,125.764706,72.66436,64.0
4,6.0,20.0,14.0,8.800000,2.080000,8.0,7.0,87.0,80.0,78.150000,...,0.0,0.0,0.0,0.0,12.0,225.0,213.0,86.350000,89.22000,12.0
